In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # EfficientNetV2 ve CvT gibi modeller iÃƒÂ§in kritik

from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau # LR Scheduler'lar iÃƒÂ§in

# UyarÃ„Â±larÃ„Â± gizlemek
warnings.filterwarnings("ignore")

# --- GLOBAL AYARLAR VE TEKRARLANABÃ„Â°LÃ„Â°RLÃ„Â°K ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Gerekli tÃƒÂ¼m kÃƒÂ¼tÃƒÂ¼phaneler ve global ayarlar yÃƒÂ¼klendi.")

In [ ]:

# --- KONFÃ„Â°GÃƒÅ“RASYON (CvT - Convolutional Vision Transformer) ---
# CvT-13 Modeli (timm kodu: cvt_13) - Denge ve performans iÃƒÂ§in tercih edilen varyant.
MODEL_NAME = 'cvt_13'

# Deney AdÃ„Â± (Takip kolaylÃ„Â±Ã„Å¸Ã„Â± iÃƒÂ§in)
EXPERIMENT_NAME = "CvT13_Baseline_MediumLarge_Run1"

# Hiperparametreler
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.00005
NUM_CLASSES = 8
DROPOUT_RATE = 0.1
# CvT iÃƒÂ§in KRÃ„Â°TÃ„Â°K: L2 dÃƒÂ¼zenlileÃ…Å¸tirme (Weight Decay) daha yÃƒÂ¼ksek ayarlanÃ„Â±r (AdamW ile kullanÃ„Â±lÃ„Â±r).
WEIGHT_DECAY = 0.05

# --- DOSYA YOLLARI ---
DATA_DIR = "../data/prepared-data"

# SonuÃƒÂ§larÃ„Â±n kaydedileceÃ„Å¸i yer
OUTPUT_DIR = f"../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihaz KontrolÃƒÂ¼
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"KayÃ„Â±t Yeri: {OUTPUT_DIR}")

In [ ]:
# ImageNet Normalize DeÃ„Å¸erleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri DÃƒÂ¶nÃƒÂ¼Ã…Å¸ÃƒÂ¼mleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri OluÃ…Å¸tur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderlarÃ„Â± OluÃ…Å¸tur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"SÃ„Â±nÃ„Â±flar: {class_names}")
print(f"EÃ„Å¸itim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

In [ ]:
def create_model(model_name: str, num_classes: int, dropout_rate: float, device: str):
    """
    Belirtilen model adÃ„Â±yla (timm) modeli oluÃ…Å¸turur, aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â±nÃ„Â± yÃƒÂ¼kler ve cihaz ÃƒÂ¼zerine taÃ…Å¸Ã„Â±r.
    """
    print(f"\nModel indiriliyor: {model_name}...")

    # pretrained=True ile ImageNet aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â±nÃ„Â± alÃ„Â±yoruz
    try:
        model = timm.create_model(model_name,
                                  pretrained=True,
                                  num_classes=num_classes,
                                  drop_rate=dropout_rate)
    except Exception as e:
        print(f"Hata: Model '{model_name}' yÃƒÂ¼klenemedi. Kontrol edin. Hata: {e}")
        return None

    model = model.to(device)
    return model

# --- KURULUM ---
# MODEL_NAME, NUM_CLASSES, DROPOUT_RATE ve DEVICE deÃ„Å¸iÃ…Å¸kenlerinin konfigÃƒÂ¼rasyon bloÃ„Å¸undan geldiÃ„Å¸i varsayÃ„Â±lÃ„Â±r.
model = create_model(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, DEVICE)

# KayÃ„Â±p Fonksiyonu
criterion = nn.CrossEntropyLoss()

# Optimizer: CvT iÃƒÂ§in kritik olan AdamW kullanÃ„Â±lÃ„Â±yor.
# WEIGHT_DECAY, daha ÃƒÂ¶nce tanÃ„Â±mladÃ„Â±Ã„Å¸Ã„Â±nÃ„Â±z CvT konfigÃƒÂ¼rasyon bloÃ„Å¸undan (0.05) geliyor.
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning Rate Scheduler: CosineAnnealingLR, Transformer tabanlÃ„Â± modeller iÃƒÂ§in en iyi performansÃ„Â± saÃ„Å¸lar.
# T_max (EPOCHS) konfigÃƒÂ¼rasyondan gelmelidir.
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("CvT Model GPU'ya yÃƒÂ¼klendi ve eÃ„Å¸itime hazÃ„Â±r.")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # DATALOADERS ve DATASET_SIZES'Ã„Â±n global olarak tanÃ„Â±mlandÃ„Â±Ã„Å¸Ã„Â±nÃ„Â± varsayÃ„Â±yoruz.

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch DÃƒÂ¶ngÃƒÂ¼sÃƒÂ¼
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Epoch Sonucu Hesaplama
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # GeÃƒÂ§miÃ…Å¸i Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # --- SCHEDULER VE MODEL KAYDI ---

            if phase == 'train':
                # CosineAnnealingLR (CvT iÃƒÂ§in ÃƒÂ¶nerilen) her eÃ„Å¸itim epoch'undan sonra adÃ„Â±m atar.
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']
                print(f"Current LR: {current_lr:.6f}")

            if phase == 'val':
                # En iyi modeli kaydet (Validasyon baÃ…Å¸arÃ„Â±sÃ„Â±na gÃƒÂ¶re)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En Ã„Â°yi Model! ({best_acc:.4f}) -> Kaydedildi ve Yolu: {save_path}")

    time_elapsed = time.time() - since
    print(f'\nEÃ„Å¸itim TamamlandÃ„Â±: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En Ã„Â°yi Validasyon DoÃ„Å¸ruluÃ„Å¸u: {best_acc:.4f}')

    # En iyi aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â± geri yÃƒÂ¼kle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history

In [ ]:
# --- BAÃ…ÂLAT ---
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
plt.figure(figsize=(14, 5))

# DoÃ„Å¸ruluk GrafiÃ„Å¸i
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title(f'{MODEL_NAME} Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# KayÃ„Â±p GrafiÃ„Å¸i
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Kaydet ve GÃƒÂ¶ster
plt.savefig(os.path.join(OUTPUT_DIR, 'training_graph.png'))
plt.show()
print(f"Grafikler kaydedildi: {os.path.join(OUTPUT_DIR, 'training_graph.png')}")

In [ ]:
print("\nTEST SETÃ„Â° DEÃ„ÂERLENDÃ„Â°RMESÃ„Â°")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 1. Classification Report (F1, Recall, Precision)
print("\nSÃ„Â±nÃ„Â±flandÃ„Â±rma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen')
plt.ylabel('GerÃƒÂ§ek')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()